# EventBridge vs Notifications API — Latency Comparison

Head-to-head comparison of **Genesys Notifications API (WebSocket)** and **EventBridge (SQS)** transcription delivery latency, using **poc-deepgram** as ground-truth audio clock.

## What We're Comparing

Both delivery paths share Stages 1–3 (audio capture → STT → endpointing). The difference is Stage 4:

```
Notifications API (WebSocket):
Speaker → [1] Audio Capture → [2] r2d2 STT → [3] Endpointing
  → [4a] WebSocket push to notifications-spike

EventBridge (SQS):
Speaker → [1] Audio Capture → [2] r2d2 STT → [3] Endpointing
  → [4b-i] EB publish → [4b-ii] EB rule → SQS enqueue → [4b-iii] Consumer polls
```

## Formula

```
true_latency = receivedAt - deepgram_audio_wall_clock_end
```

Where `receivedAt` is from notifications-spike (WebSocket) or sqs_consumer (SQS poll), and `deepgram_audio_wall_clock_end` is the ground-truth wall-clock time from poc-deepgram.

See `docs/eventbridge_comparison_plan.md` for the full implementation plan.

---
## Module 1: Setup & Configuration

In [ ]:
import json
import statistics
import sys
import warnings
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_style("whitegrid")
warnings.filterwarnings("ignore", category=FutureWarning)

REPO_ROOT = Path("..").resolve()
sys.path.insert(0, str(REPO_ROOT))

from scripts.correlate_latency import (
    correlate,
    correlate_eventbridge,
    load_deepgram_session,
    load_genesys_conversation,
    load_eventbridge_conversation,
    match_utterances,
    compute_latency,
    CorrelationResult,
)

# === DIRECTORIES ===
DEEPGRAM_RESULTS_DIR = (REPO_ROOT / ".." / "poc-deepgram" / "results").resolve()
NOTIF_EVENTS_DIR = (REPO_ROOT / "conversation_events").resolve()
EB_EVENTS_DIR = (REPO_ROOT / "EventBridge" / "conversation_events").resolve()

OUTPUT_DIR = REPO_ROOT / "analysis_results" / "cross_system_eb"
SAVE_DPI = 300
SIMILARITY_THRESHOLD = 0.55
NUM_RECENT = 6  # Number of most recent sessions to correlate

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Repo root: {REPO_ROOT}")
print(f"Output dir: {OUTPUT_DIR}")
print(f"Deepgram results: {DEEPGRAM_RESULTS_DIR}")
print(f"Notifications events: {NOTIF_EVENTS_DIR}")
print(f"EventBridge events: {EB_EVENTS_DIR}")

---
## Module 2: Load & Auto-Match Files

Match files from all 3 sources:
- **Deepgram ↔ Notifications**: by overlapping time windows
- **Deepgram ↔ EventBridge**: by overlapping time windows
- **Notifications ↔ EventBridge**: by conversation ID (same UUID in both directories)

In [ ]:
# List available files from all 3 systems
dg_files = sorted(DEEPGRAM_RESULTS_DIR.glob("*.json"), key=lambda f: f.stat().st_mtime, reverse=True)
notif_files = sorted(NOTIF_EVENTS_DIR.glob("*.jsonl"), key=lambda f: f.stat().st_mtime, reverse=True)
eb_files = sorted(EB_EVENTS_DIR.glob("*.jsonl"), key=lambda f: f.stat().st_mtime, reverse=True)

print(f"Available Deepgram sessions: {len(dg_files)}")
for i, f in enumerate(dg_files[:NUM_RECENT + 2]):
    data = json.loads(f.read_text())
    session = data.get("session", {})
    print(f"  [{i}] {f.name}  ({session.get('started_at', '?')} — {session.get('duration_seconds', '?')}s)")

print(f"\nAvailable Notifications conversations: {len(notif_files)}")
for i, f in enumerate(notif_files[:NUM_RECENT + 2]):
    lines = f.read_text().strip().splitlines()
    if lines:
        first = json.loads(lines[0])
        ts = datetime.fromtimestamp(first["receivedAt"], tz=timezone.utc)
        print(f"  [{i}] {f.name}  ({ts:%Y-%m-%d %H:%M} — {len(lines)} events)")

print(f"\nAvailable EventBridge conversations: {len(eb_files)}")
for i, f in enumerate(eb_files[:NUM_RECENT + 2]):
    lines = f.read_text().strip().splitlines()
    if lines:
        first = json.loads(lines[0])
        ts = datetime.fromtimestamp(first["receivedAt"], tz=timezone.utc)
        print(f"  [{i}] {f.name}  ({ts:%Y-%m-%d %H:%M} — {len(lines)} events)")

In [ ]:
def get_dg_time_range(path):
    """Extract session start/end times from a Deepgram session JSON."""
    data = json.loads(path.read_text())
    session = data.get("session", {})
    started = session.get("started_at", "")
    ended = session.get("ended_at", "")
    start_ts = datetime.fromisoformat(started).timestamp() if started else 0
    end_ts = datetime.fromisoformat(ended).timestamp() if ended else float("inf")
    return start_ts, end_ts


def get_jsonl_time_range(path):
    """Extract first/last receivedAt from a JSONL file."""
    lines = path.read_text().strip().splitlines()
    if not lines:
        return 0, 0
    timestamps = [json.loads(line)["receivedAt"] for line in lines]
    return min(timestamps), max(timestamps)


# Take the N most recent files
dg_recent = dg_files[:NUM_RECENT]
notif_recent = notif_files[:NUM_RECENT]
eb_recent = eb_files[:NUM_RECENT]

# Match Deepgram ↔ Notifications by time overlap
notif_matched = []
used_notif = set()
for dg_path in dg_recent:
    dg_start, dg_end = get_dg_time_range(dg_path)
    best_idx, best_overlap = None, 0
    for j, nf in enumerate(notif_recent):
        if j in used_notif:
            continue
        nf_start, nf_end = get_jsonl_time_range(nf)
        overlap = max(0, min(dg_end, nf_end) - max(dg_start, nf_start))
        if overlap > best_overlap:
            best_overlap = overlap
            best_idx = j
    if best_idx is not None:
        notif_matched.append((dg_path, notif_recent[best_idx]))
        used_notif.add(best_idx)

# Match Notifications ↔ EventBridge by conversation ID
eb_by_conv_id = {f.stem: f for f in eb_recent}

# Build triple matches: (deepgram, notifications, eventbridge)
triple_matched = []
for dg_path, notif_path in notif_matched:
    conv_id = notif_path.stem
    eb_path = eb_by_conv_id.get(conv_id)
    triple_matched.append((dg_path, notif_path, eb_path))

print(f"\nMatched {len(triple_matched)} session triple(s):")
for dg_p, nf_p, eb_p in triple_matched:
    eb_status = eb_p.name if eb_p else "MISSING"
    print(f"  DG: {dg_p.name}")
    print(f"    Notif: {nf_p.name}")
    print(f"    EB:    {eb_status}")

---
## Module 2.5: Data Quality Validation

Verify both delivery paths captured the same conversations with similar event counts.

In [ ]:
print("DATA QUALITY CHECK")
print("=" * 70)

quality_issues = []

for dg_p, nf_p, eb_p in triple_matched:
    conv_id = nf_p.stem
    print(f"\nConversation: {conv_id}")

    # Count Notifications events
    nf_lines = nf_p.read_text().strip().splitlines()
    nf_total = len(nf_lines)
    nf_final = sum(
        1 for line in nf_lines
        if json.loads(line).get("transcript", {}).get("isFinal", False)
    )

    # Count EventBridge events
    if eb_p and eb_p.exists():
        eb_lines = eb_p.read_text().strip().splitlines()
        eb_total = len(eb_lines)
        eb_final = 0
        for line in eb_lines:
            raw = json.loads(line)
            for t in raw.get("transcripts", []):
                if t.get("isFinal", False):
                    eb_final += 1
        print(f"  Notifications: {nf_total} total events, {nf_final} isFinal")
        print(f"  EventBridge:   {eb_total} total events, {eb_final} isFinal")

        if abs(nf_final - eb_final) > max(2, nf_final * 0.2):
            msg = f"  ⚠️  isFinal count mismatch: Notif={nf_final}, EB={eb_final}"
            print(msg)
            quality_issues.append(msg)
        else:
            print(f"  ✓ Event counts are consistent")
    else:
        msg = f"  ⚠️  EventBridge data MISSING for {conv_id}"
        print(msg)
        quality_issues.append(msg)

if quality_issues:
    print(f"\n{'!' * 70}")
    print(f"WARNING: {len(quality_issues)} quality issue(s) detected.")
    for issue in quality_issues:
        print(issue)
    print(f"{'!' * 70}")
else:
    print(f"\n✓ All conversations have consistent data across both paths.")

---
## Module 3: Correlate Both Paths with Deepgram Ground Truth

In [ ]:
# Correlate Notifications path
notif_results = []
for dg_path, nf_path, eb_path in triple_matched:
    pair_results = correlate(dg_path, nf_path, similarity_threshold=SIMILARITY_THRESHOLD)
    notif_results.extend(pair_results)
    print(f"  Notif: {dg_path.name} <-> {nf_path.name}: {len(pair_results)} matched")

# Correlate EventBridge path
eb_results = []
for dg_path, nf_path, eb_path in triple_matched:
    if eb_path and eb_path.exists():
        pair_results = correlate_eventbridge(
            dg_path, eb_path, similarity_threshold=SIMILARITY_THRESHOLD
        )
        eb_results.extend(pair_results)
        print(f"  EB:    {dg_path.name} <-> {eb_path.name}: {len(pair_results)} matched")
    else:
        print(f"  EB:    {dg_path.name} — SKIPPED (no EB data)")

print(f"\nTotal Notifications matched pairs: {len(notif_results)}")
print(f"Total EventBridge matched pairs:   {len(eb_results)}")

# Build DataFrames
def results_to_df(results, source_label):
    if not results:
        return pd.DataFrame()
    rows = [
        {
            "source": source_label,
            "deepgram_transcript": r.deepgram_transcript,
            "genesys_transcript": r.genesys_transcript,
            "audio_wall_clock_end": r.audio_wall_clock_end,
            "genesys_received_at": r.genesys_received_at,
            "true_latency_s": r.true_latency_s,
            "true_latency_ms": r.true_latency_ms,
            "channel": r.channel,
            "similarity": r.similarity,
        }
        for r in results
    ]
    df = pd.DataFrame(rows)
    df["received_dt"] = pd.to_datetime(df["genesys_received_at"], unit="s", utc=True)
    df["audio_dt"] = pd.to_datetime(df["audio_wall_clock_end"], unit="s", utc=True)
    return df

df_notif = results_to_df(notif_results, "Notifications")
df_eb = results_to_df(eb_results, "EventBridge")
df_all = pd.concat([df_notif, df_eb], ignore_index=True)

print(f"\nNotifications DataFrame: {len(df_notif)} rows")
print(f"EventBridge DataFrame:   {len(df_eb)} rows")

---
## Module 4: Summary Statistics (per path)

In [ ]:
def print_latency_summary(df, label):
    """Print summary statistics for a single delivery path."""
    if df.empty:
        print(f"\n{label}: No data")
        return
    lat = df["true_latency_ms"]
    print(f"\n{'=' * 60}")
    print(f"{label} — LATENCY SUMMARY")
    print(f"{'=' * 60}")
    print(f"  Matched pairs:    {len(df)}")
    print(f"  Mean latency:     {lat.mean():.0f} ms")
    print(f"  Median latency:   {lat.median():.0f} ms")
    print(f"  Std deviation:    {lat.std():.0f} ms")
    print(f"  Min latency:      {lat.min():.0f} ms")
    print(f"  Max latency:      {lat.max():.0f} ms")
    print()
    for p in [0.50, 0.75, 0.90, 0.95, 0.99]:
        print(f"  p{int(p*100):02d}:             {lat.quantile(p):.0f} ms")
    channels = df["channel"].unique()
    if len(channels) > 1:
        print(f"\n  By Channel:")
        for ch in sorted(channels):
            ch_data = df[df["channel"] == ch]["true_latency_ms"]
            print(f"    {ch}: median={ch_data.median():.0f}ms, mean={ch_data.mean():.0f}ms, n={len(ch_data)}")


print_latency_summary(df_notif, "Notifications API (WebSocket)")
print_latency_summary(df_eb, "EventBridge (SQS)")

---
## Module 4.5: Self-Reported Latency (Anchor-Event Method)

Recompute Genesys self-reported latency for the same 6 calls using the anchor-event method from `latency_analysis-01-RESULTS.ipynb`. This method is **delivery-path-independent** — the constant delivery overhead cancels out via the anchor event.

```
conversation_start = min(receivedAt - (offsetMs + durationMs) / 1000)  across all events
latency = receivedAt - (conversation_start + (offsetMs + durationMs) / 1000)
```

In [ ]:
def calculate_conversation_latency(events):
    """Calculate self-reported latency for a list of Notifications JSONL events.

    Uses the anchor-event method: the event with the smallest
    (receivedAt - audio_end_offset) defines conversation_start, and all
    other events' latency is measured relative to that anchor.

    Returns list of dicts with latency info, or empty list if < 2 events.
    """
    final_events = []
    for event in events:
        t = event.get("transcript", {})
        if not t.get("isFinal", False):
            continue
        alts = t.get("alternatives", [])
        if not alts:
            continue
        alt = alts[0]
        offset_ms = alt.get("offsetMs", 0)
        duration_ms = alt.get("durationMs", 0)
        received_at = event["receivedAt"]
        audio_end_s = (offset_ms + duration_ms) / 1000.0
        final_events.append({
            "receivedAt": received_at,
            "audio_end_s": audio_end_s,
            "transcript": alt.get("transcript", ""),
            "channel": t.get("channel", "UNKNOWN"),
            "offsetMs": offset_ms,
            "durationMs": duration_ms,
        })

    if len(final_events) < 2:
        return []

    # Anchor: event that minimizes (receivedAt - audio_end_s)
    conversation_start = min(e["receivedAt"] - e["audio_end_s"] for e in final_events)

    results = []
    for e in final_events:
        expected_arrival = conversation_start + e["audio_end_s"]
        latency_s = e["receivedAt"] - expected_arrival
        results.append({
            "latency_s": latency_s,
            "latency_ms": latency_s * 1000,
            "transcript": e["transcript"],
            "channel": e["channel"],
        })
    return results


# Compute self-reported latency for the NEW Notifications data
self_reported_all = []
for _, nf_path, _ in triple_matched:
    lines = nf_path.read_text().strip().splitlines()
    events = [json.loads(line) for line in lines if line.strip()]
    conv_latencies = calculate_conversation_latency(events)
    self_reported_all.extend(conv_latencies)
    if conv_latencies:
        meds = statistics.median([r["latency_ms"] for r in conv_latencies])
        print(f"  {nf_path.stem}: {len(conv_latencies)} utterances, median {meds:.0f}ms")

if self_reported_all:
    sr_latencies_ms = [r["latency_ms"] for r in self_reported_all]
    print(f"\nSelf-Reported Summary (anchor-event method, {len(sr_latencies_ms)} utterances):")
    print(f"  Median: {statistics.median(sr_latencies_ms):.0f} ms")
    print(f"  Mean:   {statistics.mean(sr_latencies_ms):.0f} ms")
    if len(sr_latencies_ms) >= 20:
        sorted_sr = sorted(sr_latencies_ms)
        p95_idx = int(len(sorted_sr) * 0.95)
        p99_idx = int(len(sorted_sr) * 0.99)
        print(f"  p95:    {sorted_sr[p95_idx]:.0f} ms")
        print(f"  p99:    {sorted_sr[p99_idx]:.0f} ms")
else:
    print("No self-reported latency data (need ≥2 isFinal events per conversation).")

---
## Module 5: Head-to-Head Comparison Table

The key deliverable. Four rows comparing delivery mechanisms:

| Row | What It Measures | Formula |
|-----|------------------|---------|
| **Notifications API** | True end-to-end via WebSocket | `notif_receivedAt - deepgram_audio_end` |
| **EventBridge (SQS)** | True end-to-end via EB→SQS | `eb_receivedAt - deepgram_audio_end` |
| **Genesys Self-Reported** | Internal processing (anchor-relative) | anchor-event method |
| **Deepgram/Audio Hook** | Deepgram Nova-3 STT latency (reference) | `server_receipt_time - audio_end` |

In [ ]:
def compute_stats(values_ms):
    """Compute summary stats for a list of latency values in ms."""
    if not values_ms:
        return {k: float("nan") for k in ["median", "mean", "p95", "p99", "min", "max", "n"]}
    s = sorted(values_ms)
    n = len(s)
    return {
        "median": statistics.median(s),
        "mean": statistics.mean(s),
        "p95": s[int(n * 0.95)] if n >= 20 else s[-1],
        "p99": s[int(n * 0.99)] if n >= 100 else s[-1],
        "min": s[0],
        "max": s[-1],
        "n": n,
    }


# Deepgram/Audio Hook latency from the session files
dg_latencies_ms = []
for dg_path, _, _ in triple_matched:
    data = json.loads(dg_path.read_text())
    for t in data.get("transcripts", []):
        lat = t.get("latency_ms")
        if lat is not None:
            dg_latencies_ms.append(lat)

# Build comparison rows
rows = {
    "Notifications API (WebSocket)": compute_stats(df_notif["true_latency_ms"].tolist()) if not df_notif.empty else compute_stats([]),
    "EventBridge (SQS)": compute_stats(df_eb["true_latency_ms"].tolist()) if not df_eb.empty else compute_stats([]),
    "Genesys Self-Reported": compute_stats(sr_latencies_ms if self_reported_all else []),
    "Deepgram/Audio Hook ¹": compute_stats(dg_latencies_ms),
}

# Print comparison table
print("\n" + "=" * 100)
print("HEAD-TO-HEAD COMPARISON TABLE")
print("=" * 100)
header = f"{'Source':<35} {'Median':>8} {'Mean':>8} {'p95':>8} {'p99':>8} {'Min':>8} {'Max':>8} {'N':>5}"
print(header)
print("-" * 100)

for label, stats in rows.items():
    print(
        f"{label:<35} {stats['median']:>7.0f}ms {stats['mean']:>7.0f}ms "
        f"{stats['p95']:>7.0f}ms {stats['p99']:>7.0f}ms "
        f"{stats['min']:>7.0f}ms {stats['max']:>7.0f}ms {stats['n']:>5.0f}"
    )

# Delta and ratio rows
notif_stats = rows["Notifications API (WebSocket)"]
eb_stats = rows["EventBridge (SQS)"]

if notif_stats["n"] > 0 and eb_stats["n"] > 0:
    print("-" * 100)
    delta_line = f"{'Delta (EB - Notif)':<35}"
    ratio_line = f"{'Ratio (EB / Notif)':<35}"
    for metric in ["median", "mean", "p95", "p99", "min", "max"]:
        d = eb_stats[metric] - notif_stats[metric]
        r = eb_stats[metric] / notif_stats[metric] if notif_stats[metric] != 0 else float("inf")
        delta_line += f" {d:>+7.0f}ms"
        ratio_line += f" {r:>7.2f}x "
    print(delta_line)
    print(ratio_line)

print("\n¹ Deepgram processes the same audio independently via BlackHole. This measures")
print("  Deepgram Nova-3 STT latency (300ms endpointing) as a reference benchmark,")
print("  not a direct comparison to Genesys delivery mechanisms.")

---
## Module 6: Visualizations

In [ ]:
# Chart 1: Distribution overlay — Notifications vs EventBridge
if not df_notif.empty and not df_eb.empty:
    fig, ax = plt.subplots(figsize=(12, 5))
    bins = np.linspace(
        min(df_all["true_latency_ms"].min(), 0),
        df_all["true_latency_ms"].max() * 1.05,
        40,
    )
    ax.hist(df_notif["true_latency_ms"], bins=bins, alpha=0.6, color="steelblue",
            edgecolor="black", linewidth=0.5, label="Notifications (WebSocket)")
    ax.hist(df_eb["true_latency_ms"], bins=bins, alpha=0.6, color="darkorange",
            edgecolor="black", linewidth=0.5, label="EventBridge (SQS)")
    ax.axvline(df_notif["true_latency_ms"].median(), color="steelblue", linestyle="--",
               linewidth=2, label=f"Notif median: {df_notif['true_latency_ms'].median():.0f}ms")
    ax.axvline(df_eb["true_latency_ms"].median(), color="darkorange", linestyle="--",
               linewidth=2, label=f"EB median: {df_eb['true_latency_ms'].median():.0f}ms")
    ax.set_xlabel("True Latency (ms)")
    ax.set_ylabel("Count")
    ax.set_title("Latency Distribution: Notifications vs EventBridge", fontsize=14, fontweight="bold")
    ax.legend()
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "distribution_overlay.png", dpi=SAVE_DPI, bbox_inches="tight")
    plt.show()
else:
    print("Skipping distribution overlay — need data from both paths.")

In [ ]:
# Chart 2: Side-by-side box plots (all 4 rows)
box_data = []
box_labels = []

if not df_notif.empty:
    box_data.append(df_notif["true_latency_ms"].values)
    box_labels.append(f"Notifications\n(n={len(df_notif)})")
if not df_eb.empty:
    box_data.append(df_eb["true_latency_ms"].values)
    box_labels.append(f"EventBridge\n(n={len(df_eb)})")
if self_reported_all:
    box_data.append(np.array(sr_latencies_ms))
    box_labels.append(f"Self-Reported\n(n={len(sr_latencies_ms)})")
if dg_latencies_ms:
    box_data.append(np.array(dg_latencies_ms))
    box_labels.append(f"Deepgram\n(n={len(dg_latencies_ms)})")

if box_data:
    fig, ax = plt.subplots(figsize=(10, 6))
    colors = ["steelblue", "darkorange", "seagreen", "mediumpurple"]
    bp = ax.boxplot(box_data, labels=box_labels, patch_artist=True,
                    showfliers=True, flierprops={"marker": ".", "alpha": 0.3})
    for patch, color in zip(bp["boxes"], colors[:len(box_data)]):
        patch.set_facecolor(color)
        patch.set_alpha(0.6)
    ax.set_ylabel("Latency (ms)")
    ax.set_title("Latency Comparison Across All Sources", fontsize=14, fontweight="bold")
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "boxplot_comparison.png", dpi=SAVE_DPI, bbox_inches="tight")
    plt.show()

In [ ]:
# Chart 3: Timeline scatter colored by source
if not df_all.empty and len(df_all) > 1:
    fig, ax = plt.subplots(figsize=(14, 6))
    source_colors = {"Notifications": "steelblue", "EventBridge": "darkorange"}
    for source, color in source_colors.items():
        mask = df_all["source"] == source
        if mask.any():
            subset = df_all[mask].sort_values("audio_dt")
            ax.scatter(subset["audio_dt"], subset["true_latency_ms"],
                       alpha=0.6, s=40, color=color, label=source)
    ax.set_xlabel("Audio Time (UTC)")
    ax.set_ylabel("True Latency (ms)")
    ax.set_title("Latency Over Time by Delivery Path", fontsize=14, fontweight="bold")
    ax.legend()
    fig.autofmt_xdate()
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "timeline_by_source.png", dpi=SAVE_DPI, bbox_inches="tight")
    plt.show()

---
## Module 7: EventBridge 3-Hop Analysis

Decompose the EventBridge pipeline latency into 3 hops using the 4 delivery timestamps:

```
genesysEventTime ──> ebTime ──> sqsSentTimestamp ──> receivedAt
      │                  │              │                   │
      └── Hop 1 ────────┘              │                   │
           Genesys → EB                 │                   │
           (second precision ⚠️)        │                   │
                         └── Hop 2 ────┘                   │
                              EB → SQS enqueue             │
                              (limited by ebTime ⚠️)       │
                                        └── Hop 3 ────────┘
                                             SQS queue → consumer poll
                                             (ms precision ✓)
```

**⚠️ Precision caveat:** `ebTime` has only second-level granularity (`2026-03-19T22:04:48Z`), while `genesysEventTime` has ms precision. Hop 1 and Hop 2 measurements have ~1s rounding error. Only Hop 3 (`sqsSentTimestamp` → `receivedAt`) has ms precision on both ends.

In [ ]:
from datetime import datetime, timezone


def parse_iso_to_epoch(iso_str):
    """Parse ISO 8601 string to epoch seconds (float)."""
    # Handle both second-level ('2026-03-19T22:04:48Z') and ms-level ('2026-03-19T22:04:48.128Z')
    dt = datetime.fromisoformat(iso_str.replace("Z", "+00:00"))
    return dt.timestamp()


# Read raw EB JSONL for hop analysis (separate from GenesysEvent pipeline)
hop_data = []
for _, _, eb_path in triple_matched:
    if not eb_path or not eb_path.exists():
        continue
    for line in eb_path.read_text().strip().splitlines():
        if not line.strip():
            continue
        raw = json.loads(line)
        genesys_event_time = parse_iso_to_epoch(raw["genesysEventTime"])
        eb_time = parse_iso_to_epoch(raw["ebTime"])
        sqs_sent_ts = raw.get("sqsSentTimestamp")
        received_at = raw["receivedAt"]

        if sqs_sent_ts is None:
            continue

        sqs_sent_s = sqs_sent_ts / 1000.0

        hop1_ms = (eb_time - genesys_event_time) * 1000  # Genesys → EB (second precision!)
        hop2_ms = (sqs_sent_s - eb_time) * 1000          # EB → SQS (limited by ebTime)
        hop3_ms = (received_at - sqs_sent_s) * 1000      # SQS → consumer (ms precision)
        total_ms = (received_at - genesys_event_time) * 1000

        hop_data.append({
            "hop1_ms": hop1_ms,
            "hop2_ms": hop2_ms,
            "hop3_ms": hop3_ms,
            "total_ms": total_ms,
        })

if hop_data:
    df_hops = pd.DataFrame(hop_data)

    print("EVENTBRIDGE 3-HOP ANALYSIS")
    print("=" * 70)
    print(f"Events analyzed: {len(df_hops)}")
    print()

    hop_labels = {
        "hop1_ms": "Hop 1: Genesys → EB (⚠️ ~1s precision)",
        "hop2_ms": "Hop 2: EB → SQS enqueue (⚠️ limited)",
        "hop3_ms": "Hop 3: SQS → Consumer poll (✓ ms)",
        "total_ms": "Total: Genesys → Consumer",
    }

    header = f"{'Hop':<45} {'Median':>8} {'Mean':>8} {'p95':>8} {'Min':>8} {'Max':>8}"
    print(header)
    print("-" * 85)
    for col, label in hop_labels.items():
        vals = df_hops[col]
        p95 = vals.quantile(0.95) if len(vals) >= 20 else vals.max()
        print(f"{label:<45} {vals.median():>7.0f}ms {vals.mean():>7.0f}ms "
              f"{p95:>7.0f}ms {vals.min():>7.0f}ms {vals.max():>7.0f}ms")

    print(f"\n⚠️  Hop 1 and Hop 2 have ~1s rounding error due to ebTime second-level precision.")
    print(f"    Only Hop 3 (sqsSentTimestamp → receivedAt) has ms precision on both ends.")
else:
    print("No EventBridge hop data available.")
    df_hops = pd.DataFrame()

In [ ]:
# Chart 4: Stacked bar chart — mean per hop
if not df_hops.empty:
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Stacked bar chart
    ax = axes[0]
    hop_means = [
        df_hops["hop1_ms"].mean(),
        df_hops["hop2_ms"].mean(),
        df_hops["hop3_ms"].mean(),
    ]
    hop_names = ["Hop 1\nGenesys→EB", "Hop 2\nEB→SQS", "Hop 3\nSQS→Consumer"]
    colors = ["#ff9999", "#66b3ff", "#99ff99"]
    bottom = 0
    for i, (mean, name, color) in enumerate(zip(hop_means, hop_names, colors)):
        ax.bar("EB Pipeline", mean, bottom=bottom, color=color, edgecolor="black",
               linewidth=0.5, label=f"{name}: {mean:.0f}ms")
        bottom += mean
    ax.set_ylabel("Latency (ms)")
    ax.set_title("Mean Latency by Hop", fontsize=12, fontweight="bold")
    ax.legend(loc="upper right")

    # Scatter: Hop 3 (most precise)
    ax = axes[1]
    ax.scatter(range(len(df_hops)), df_hops["hop3_ms"], alpha=0.5, s=20, color="#99ff99",
               edgecolors="black", linewidth=0.3)
    ax.axhline(df_hops["hop3_ms"].median(), color="green", linestyle="--",
               label=f"Median: {df_hops['hop3_ms'].median():.0f}ms")
    ax.set_xlabel("Event Index")
    ax.set_ylabel("Hop 3 Latency (ms)")
    ax.set_title("Hop 3: SQS → Consumer (ms precision)", fontsize=12, fontweight="bold")
    ax.legend()

    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "hop_analysis.png", dpi=SAVE_DPI, bbox_inches="tight")
    plt.show()

---
## Module 8: Matched Pairs Detail Tables

In [ ]:
def print_matched_pairs(df, label):
    """Print matched pairs detail table."""
    if df.empty:
        print(f"\n{label}: No data")
        return
    print(f"\n{'=' * 120}")
    print(f"{label} — MATCHED PAIRS")
    print(f"{'=' * 120}")
    print(f"{'Latency':>10}  {'Sim':>5}  {'Ch':>8}  {'Deepgram Transcript':<40}  {'Genesys Transcript':<40}")
    print("-" * 120)
    for _, row in df.iterrows():
        dg_t = row["deepgram_transcript"][:38] + (".." if len(row["deepgram_transcript"]) > 38 else "")
        gn_t = row["genesys_transcript"][:38] + (".." if len(row["genesys_transcript"]) > 38 else "")
        print(f"{row['true_latency_ms']:>8.0f}ms  {row['similarity']:>5.2f}  {row['channel']:>8}  {dg_t:<40}  {gn_t:<40}")


print_matched_pairs(df_notif, "Notifications API (WebSocket)")
print_matched_pairs(df_eb, "EventBridge (SQS)")

---
## Module 9: Export Results

In [ ]:
from scripts.correlate_latency import export_csv

# Export CSVs
if notif_results:
    export_csv(notif_results, OUTPUT_DIR / "notif_correlation.csv")
if eb_results:
    export_csv(eb_results, OUTPUT_DIR / "eb_correlation.csv")

# Export head-to-head comparison JSON
comparison = {
    "session_triples": [
        {
            "deepgram": dg.name,
            "notifications": nf.name,
            "eventbridge": eb.name if eb else None,
        }
        for dg, nf, eb in triple_matched
    ],
    "similarity_threshold": SIMILARITY_THRESHOLD,
    "notifications": compute_stats(df_notif["true_latency_ms"].tolist()) if not df_notif.empty else {},
    "eventbridge": compute_stats(df_eb["true_latency_ms"].tolist()) if not df_eb.empty else {},
    "self_reported": compute_stats(sr_latencies_ms) if self_reported_all else {},
    "deepgram_audio_hook": compute_stats(dg_latencies_ms) if dg_latencies_ms else {},
}

# Add hop analysis if available
if not df_hops.empty:
    comparison["hop_analysis"] = {
        "hop1_genesys_to_eb": compute_stats(df_hops["hop1_ms"].tolist()),
        "hop2_eb_to_sqs": compute_stats(df_hops["hop2_ms"].tolist()),
        "hop3_sqs_to_consumer": compute_stats(df_hops["hop3_ms"].tolist()),
        "total": compute_stats(df_hops["total_ms"].tolist()),
        "caveat": "Hop 1 and Hop 2 have ~1s rounding error due to ebTime second-level precision.",
    }

comparison_path = OUTPUT_DIR / "head_to_head_comparison.json"
comparison_path.write_text(json.dumps(comparison, indent=2, default=float))

# Export summary JSON (same structure as notebook 01)
for label, df_source, filename in [
    ("Notifications", df_notif, "notif_summary.json"),
    ("EventBridge", df_eb, "eb_summary.json"),
]:
    if not df_source.empty:
        summary = {
            "source": label,
            "matched_pairs": len(df_source),
            "latency_ms": compute_stats(df_source["true_latency_ms"].tolist()),
            "mean_similarity": round(df_source["similarity"].mean(), 3),
        }
        (OUTPUT_DIR / filename).write_text(json.dumps(summary, indent=2, default=float))

print(f"\nExported to {OUTPUT_DIR}:")
for f in sorted(OUTPUT_DIR.iterdir()):
    print(f"  {f.name}")